In [1]:
import pandas as pd
import numpy as np

# Load datasets
wildfire = pd.read_csv('../data/wildfire_weather.csv', low_memory=False)
insurance = pd.read_csv('../data/insurance_fire_census_weather_raw.csv', low_memory=False)

# Derive year from year_month for weather rows
wildfire['derived_year'] = wildfire['year_month'].str[:4].astype(float)

# Separate fire events
fire_events = wildfire[wildfire['FIRE_NAME'].notna()].copy()

print(f"Wildfire: {wildfire.shape}, Insurance: {insurance.shape}")
print("Ready to build features.")

Wildfire: (125476, 31), Insurance: (47033, 76)
Ready to build features.


In [2]:
# === BASE TABLE: one row per zip per year (2018-2021 for training) ===

# All zip-year combos from weather data (2018-2021 only)
base = wildfire[wildfire['derived_year'].isin([2018, 2019, 2020, 2021])][['zip', 'derived_year']].dropna().drop_duplicates()
base = base.rename(columns={'derived_year': 'Year'})
print(f"Base table: {len(base)} zip-year rows")

# === TARGET: did a fire occur in this zip-year? ===
fire_zip_year = fire_events[['zip', 'Year']].dropna().drop_duplicates()
fire_zip_year['had_fire'] = 1
base = base.merge(fire_zip_year, on=['zip', 'Year'], how='left')
base['had_fire'] = base['had_fire'].fillna(0).astype(int)

# === FEATURE 1: Fire history from PRIOR years ===
# For each zip-year, count fires in all previous years
fire_with_acres = fire_events[['zip', 'Year', 'GIS_ACRES']].dropna()

def get_prior_fire_features(row):
    prior = fire_with_acres[(fire_with_acres['zip'] == row['zip']) & (fire_with_acres['Year'] < row['Year'])]
    return pd.Series({
        'prior_fire_count': len(prior),
        'prior_total_acres': prior['GIS_ACRES'].sum(),
        'prior_max_acres': prior['GIS_ACRES'].max() if len(prior) > 0 else 0,
        'had_prior_fire': 1 if len(prior) > 0 else 0
    })

print("Building fire history features (this may take a minute)...")
fire_hist = base.apply(get_prior_fire_features, axis=1)
base = pd.concat([base, fire_hist], axis=1)

# Log-transform acres (highly skewed)
base['prior_total_acres_log'] = np.log1p(base['prior_total_acres'])
base['prior_max_acres_log'] = np.log1p(base['prior_max_acres'])

print(f"\nFire history features built.")
print(base[['had_fire', 'prior_fire_count', 'prior_total_acres', 'had_prior_fire']].describe().round(2))

Base table: 10380 zip-year rows
Building fire history features (this may take a minute)...

Fire history features built.
       had_fire  prior_fire_count  prior_total_acres  had_prior_fire
count  10380.00          10380.00           10380.00        10380.00
mean       0.08              0.20             554.88            0.09
std        0.28              0.89            9249.82            0.29
min        0.00              0.00               0.00            0.00
25%        0.00              0.00               0.00            0.00
50%        0.00              0.00               0.00            0.00
75%        0.00              0.00               0.00            0.00
max        1.00             27.00          396894.52            1.00


In [3]:
# === FEATURE 2: Weather features (aggregated to zip-year) ===

weather = wildfire[
    (wildfire['derived_year'].isin([2018, 2019, 2020, 2021])) &
    (wildfire['avg_tmax_c'].notna())
].copy()

# Extract month for fire-season features
weather['month'] = weather['year_month'].str[5:7].astype(int)

# Full year aggregates
weather_year = weather.groupby(['zip', 'derived_year']).agg(
    mean_tmax=('avg_tmax_c', 'mean'),
    max_tmax=('avg_tmax_c', 'max'),
    mean_tmin=('avg_tmin_c', 'mean'),
    min_tmin=('avg_tmin_c', 'min'),
    total_precip=('tot_prcp_mm', 'sum'),
    mean_precip=('tot_prcp_mm', 'mean'),
    min_precip=('tot_prcp_mm', 'min'),
    temp_range=('avg_tmax_c', lambda x: x.max() - x.min()),
).reset_index().rename(columns={'derived_year': 'Year'})

# Fire season (June-October) aggregates
fire_season = weather[weather['month'].between(6, 10)]
fire_season_agg = fire_season.groupby(['zip', 'derived_year']).agg(
    fire_season_mean_tmax=('avg_tmax_c', 'mean'),
    fire_season_max_tmax=('avg_tmax_c', 'max'),
    fire_season_total_precip=('tot_prcp_mm', 'sum'),
).reset_index().rename(columns={'derived_year': 'Year'})

# Merge both
weather_features = weather_year.merge(fire_season_agg, on=['zip', 'Year'], how='left')

# Merge into base
base = base.merge(weather_features, on=['zip', 'Year'], how='left')
print(f"After weather features: {base.shape}")
print(f"Weather nulls: {base['mean_tmax'].isna().sum()} / {len(base)}")

After weather features: (10380, 20)
Weather nulls: 8 / 10380


In [4]:
# === FEATURE 3: Insurance + Census features ===

# Select useful columns from insurance
ins_features = insurance[[
    'Year', 'ZIP', 'Avg Fire Risk Score', 'Avg PPC',
    'Earned Exposure', 'Earned Premium',
    'Cov A Amount Weighted Avg', 'Cov C Amount Weighted Avg',
    'CAT Cov A Fire -  Incurred Losses', 'CAT Cov A Fire -  Number of Claims',
    'Non-CAT Cov A Fire -  Incurred Losses', 'Non-CAT Cov A Fire -  Number of Claims',
    'Number of High Fire Risk Exposure', 'Number of Very High Fire Risk Exposure',
    'Number of Low Fire Risk Exposure', 'Number of Moderate Fire Risk Exposure',
    'Number of Negligible Fire Risk Exposure',
    'total_population', 'median_income', 'total_housing_units',
    'average_household_size', 'educational_attainment_bachelor_or_higher',
    'poverty_status', 'housing_value', 'housing_vacancy_number',
    'median_monthly_housing_costs', 'owner_occupied_housing_units',
    'renter_occupied_housing_units'
]].copy()

# Rename ZIP to zip for merge
ins_features = ins_features.rename(columns={'ZIP': 'zip'})
ins_features['zip'] = ins_features['zip'].astype(float)

# Aggregate insurance to zip-year level (some zips have multiple rows per year)
ins_agg = ins_features.groupby(['zip', 'Year']).agg({
    'Avg Fire Risk Score': 'mean',
    'Avg PPC': 'mean',
    'Earned Exposure': 'sum',
    'Earned Premium': 'sum',
    'Cov A Amount Weighted Avg': 'mean',
    'Cov C Amount Weighted Avg': 'mean',
    'CAT Cov A Fire -  Incurred Losses': 'sum',
    'CAT Cov A Fire -  Number of Claims': 'sum',
    'Non-CAT Cov A Fire -  Incurred Losses': 'sum',
    'Non-CAT Cov A Fire -  Number of Claims': 'sum',
    'Number of High Fire Risk Exposure': 'sum',
    'Number of Very High Fire Risk Exposure': 'sum',
    'Number of Low Fire Risk Exposure': 'sum',
    'Number of Moderate Fire Risk Exposure': 'sum',
    'Number of Negligible Fire Risk Exposure': 'sum',
    'total_population': 'first',
    'median_income': 'first',
    'total_housing_units': 'first',
    'average_household_size': 'first',
    'educational_attainment_bachelor_or_higher': 'first',
    'poverty_status': 'first',
    'housing_value': 'first',
    'housing_vacancy_number': 'first',
    'median_monthly_housing_costs': 'first',
    'owner_occupied_housing_units': 'first',
    'renter_occupied_housing_units': 'first',
}).reset_index()

ins_agg['Year'] = ins_agg['Year'].astype(float)

# Merge into base
base = base.merge(ins_agg, on=['zip', 'Year'], how='left')
print(f"After insurance/census features: {base.shape}")
print(f"Fire risk score nulls: {base['Avg Fire Risk Score'].isna().sum()} / {len(base)}")
print(f"Census (population) nulls: {base['total_population'].isna().sum()} / {len(base)}")

After insurance/census features: (10380, 46)
Fire risk score nulls: 2289 / 10380
Census (population) nulls: 3262 / 10380


In [5]:
# === FEATURE 4: Backfill PPC from 2018/2019 ===

# Get most recent PPC per zip
ppc = insurance[insurance['Avg PPC'].notna()][['ZIP', 'Year', 'Avg PPC']].copy()
ppc = ppc.sort_values('Year', ascending=False).drop_duplicates('ZIP', keep='first')
ppc = ppc.rename(columns={'ZIP': 'zip', 'Avg PPC': 'ppc_backfilled'})
ppc['zip'] = ppc['zip'].astype(float)

base = base.merge(ppc[['zip', 'ppc_backfilled']], on='zip', how='left')

# Where Avg PPC is null, use backfilled value
base['Avg PPC'] = base['Avg PPC'].fillna(base['ppc_backfilled'])
base = base.drop(columns=['ppc_backfilled'])

print(f"PPC nulls after backfill: {base['Avg PPC'].isna().sum()} / {len(base)}")

PPC nulls after backfill: 2284 / 10380


In [6]:
# === Summary of feature matrix ===
print(f"Final feature matrix: {base.shape}")
print(f"Target balance: {base['had_fire'].value_counts().to_dict()}")
print(f"\n=== Null counts per column ===")
null_counts = base.isnull().sum()
print(null_counts[null_counts > 0].sort_values(ascending=False).to_string())
print(f"\nColumns with zero nulls: {(null_counts == 0).sum()}")

# Save for next notebook
base.to_csv('../data/feature_matrix.csv', index=False)
print("\nSaved to data/feature_matrix.csv")

Final feature matrix: (10380, 46)
Target balance: {0: 9500, 1: 880}

=== Null counts per column ===
median_income                                4008
housing_value                                3998
median_monthly_housing_costs                 3932
average_household_size                       3547
renter_occupied_housing_units                3262
educational_attainment_bachelor_or_higher    3262
total_population                             3262
total_housing_units                          3262
poverty_status                               3262
housing_vacancy_number                       3262
owner_occupied_housing_units                 3262
Non-CAT Cov A Fire -  Incurred Losses        2289
Number of Negligible Fire Risk Exposure      2289
Number of Moderate Fire Risk Exposure        2289
Number of Low Fire Risk Exposure             2289
Number of Very High Fire Risk Exposure       2289
Number of High Fire Risk Exposure            2289
Non-CAT Cov A Fire -  Number of Claims       2289


In [7]:
# === Handle missing values ===

base = pd.read_csv('../data/feature_matrix.csv')
print(f"Loaded: {base.shape}")

# 1. Weather nulls (only 8): fill with zip-level mean across years
weather_cols = ['mean_tmax', 'max_tmax', 'mean_tmin', 'min_tmin', 'total_precip', 
                'mean_precip', 'min_precip', 'temp_range', 'fire_season_mean_tmax',
                'fire_season_max_tmax', 'fire_season_total_precip']
for col in weather_cols:
    base[col] = base.groupby('zip')[col].transform(lambda x: x.fillna(x.mean()))
    # If still null (zip has no weather at all), fill with global mean
    base[col] = base[col].fillna(base[col].mean())

print(f"Weather nulls remaining: {base[weather_cols].isnull().sum().sum()}")

# 2. Insurance/census nulls: fill with 0 for claims/exposure, median for scores/census
# Claims and exposure: 0 means "no insurance activity" which is a valid signal
zero_fill_cols = [
    'Earned Exposure', 'Earned Premium',
    'CAT Cov A Fire -  Incurred Losses', 'CAT Cov A Fire -  Number of Claims',
    'Non-CAT Cov A Fire -  Incurred Losses', 'Non-CAT Cov A Fire -  Number of Claims',
    'Number of High Fire Risk Exposure', 'Number of Very High Fire Risk Exposure',
    'Number of Low Fire Risk Exposure', 'Number of Moderate Fire Risk Exposure',
    'Number of Negligible Fire Risk Exposure',
]
for col in zero_fill_cols:
    base[col] = base[col].fillna(0)

# Scores and continuous values: fill with median (more robust than mean for skewed data)
median_fill_cols = [
    'Avg Fire Risk Score', 'Avg PPC',
    'Cov A Amount Weighted Avg', 'Cov C Amount Weighted Avg',
    'total_population', 'median_income', 'total_housing_units',
    'average_household_size', 'educational_attainment_bachelor_or_higher',
    'poverty_status', 'housing_value', 'housing_vacancy_number',
    'median_monthly_housing_costs', 'owner_occupied_housing_units',
    'renter_occupied_housing_units'
]
for col in median_fill_cols:
    base[col] = base[col].fillna(base[col].median())

# 3. Verify no nulls remain
remaining_nulls = base.isnull().sum().sum()
print(f"Total nulls remaining: {remaining_nulls}")

# 4. Check final shape and columns
print(f"\nFinal clean matrix: {base.shape}")
print(f"Target: {base['had_fire'].value_counts().to_dict()}")
print(f"\nFeature columns ({base.shape[1] - 3} features + zip, Year, target):")
for col in base.columns:
    print(f"  {col}")

# Save clean version
base.to_csv('../data/feature_matrix_clean.csv', index=False)
print("\nSaved to data/feature_matrix_clean.csv")

Loaded: (10380, 46)
Weather nulls remaining: 0
Total nulls remaining: 0

Final clean matrix: (10380, 46)
Target: {0: 9500, 1: 880}

Feature columns (43 features + zip, Year, target):
  zip
  Year
  had_fire
  prior_fire_count
  prior_total_acres
  prior_max_acres
  had_prior_fire
  prior_total_acres_log
  prior_max_acres_log
  mean_tmax
  max_tmax
  mean_tmin
  min_tmin
  total_precip
  mean_precip
  min_precip
  temp_range
  fire_season_mean_tmax
  fire_season_max_tmax
  fire_season_total_precip
  Avg Fire Risk Score
  Avg PPC
  Earned Exposure
  Earned Premium
  Cov A Amount Weighted Avg
  Cov C Amount Weighted Avg
  CAT Cov A Fire -  Incurred Losses
  CAT Cov A Fire -  Number of Claims
  Non-CAT Cov A Fire -  Incurred Losses
  Non-CAT Cov A Fire -  Number of Claims
  Number of High Fire Risk Exposure
  Number of Very High Fire Risk Exposure
  Number of Low Fire Risk Exposure
  Number of Moderate Fire Risk Exposure
  Number of Negligible Fire Risk Exposure
  total_population
  median